In [2]:
import sys

from pathlib import Path

parent_folder = Path.cwd().parent
if str(parent_folder) not in sys.path:
    sys.path.append(str(parent_folder))
    
import warnings
warnings.filterwarnings('ignore')
import numpy as np

import simulation_functional as sim
import xobjects as xo

In [3]:
def aa_to_canon(line, sqrt_action, phase, axis='x'):
    tw = line.twiss(method='4d', freeze_longitudinal=True)
    if axis=='x':
        beta = tw.betx[0]
        alpha = tw.alfx[0]
    elif axis=='y':
        beta = tw.bety[0]
        alpha = tw.alfy[0]
        
    q_norm = sqrt_action * np.cos(phase)
    p_norm = -sqrt_action * np.sin(phase)
    q = q_norm * np.sqrt(beta)
    p = (p_norm - alpha * q_norm) / np.sqrt(beta)
    return q, p

In [ ]:
def run_all(co_dist=True, indirect=True, n_turns=1000):
    line = sim.build_line(folder=Path('../Lattice_Files/02_Aperture_Lattice_res/'))

    if co_dist==True:
        s_list = np.linspace(0, line.get_length(), 13*2, endpoint=False)
        strength_list = 5e-3 * np.cos(13 * 2*np.pi * s_list / line.get_length())
        line = sim.add_dipole(line, strength=strength_list, s=s_list, mode='normal')
    
    phase_list = np.linspace(0, 2*np.pi, 40, endpoint=False)
    sqrt_Jx_list = np.linspace(0, 0.03, 40)
    sqrt_Jx_grid, phase_grid = np.meshgrid(sqrt_Jx_list, phase_list)
    x, px = aa_to_canon(line, sqrt_Jx_grid.flatten(), phase_grid.flatten())
    y = 0.0001
    py = 0
    tw = line.twiss(method='4d', freeze_longitudinal=True)
    particles = line.build_particles(
        x=x+tw.x[0],
        y=y+tw.y[0],
        px=px+tw.px[0],
        py=py+tw.py[0],
        method='4d', freeze_longitudinal=True)
    
    line = sim.add_monitors(line, particles, n_monitors=1, n_turns=n_turns)
    twiss = line.twiss(method='4d', freeze_longitudinal=True)

    if indirect!='none':
        line, beam_df, aper_df = sim.add_spacecharge(line, beam_intensity=5e12,
                                                     n_interactions=100, 
                                                     gemitt=60e-6, indirect=indirect)
    else:
        line, beam_df, aper_df = sim.add_spacecharge(line, beam_intensity=0,
                                             n_interactions=100, 
                                             gemitt=60e-6, indirect=indirect)
    
    line.build_tracker(_context=xo.ContextCpu(omp_num_threads=10))
    line.track(particles, num_turns=n_turns, time=True)
    sim.save(line, particles, twiss, 
             filename=f'data/co_dist:{co_dist}_indirect:{indirect}', 
             aper_df=aper_df, beam_df=beam_df)

In [20]:
for co_dist in [False, True]:
    for indirect in [False, True]:
        run_all(co_dist=co_dist, indirect=indirect)

Converting sequence "synchrotron":   0%|          | 0/1990 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Slicing line:   0%|          | 0/3980 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
File saved at data/co_dist:False_indirect:none


Converting sequence "synchrotron":   0%|          | 0/1990 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/3980 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Slicing line:   0%|          | 0/4126 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
File saved at data/co_dist:True_indirect:none
